# 

In [38]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from dataset_ood_download import get_data_list
from augment_dataset import mol2graph

In [14]:
import pickle

def load_mol_set(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

In [2]:
forward_train = pd.read_parquet('/data/data/Orderly/orderly_forward_train.parquet', engine = 'pyarrow')
forward_test = pd.read_parquet('/data/data/Orderly/orderly_forward_test.parquet', engine = 'pyarrow')

retro_train = pd.read_parquet('/data/data/Orderly/orderly_retro_train.parquet', engine = 'pyarrow')
retro_test = pd.read_parquet('/data/data/Orderly/orderly_retro_test.parquet', engine = 'pyarrow')

forward_df = pd.concat([forward_train, forward_test], axis=0)
retro_df = pd.concat([retro_train, retro_test], axis=0)

In [3]:
forward_df

,original_index,agent_000,agent_001,agent_002,date_of_experiment,extracted_from_file,grant_date,is_mapped,procedure_details,product_000,...,reactant_001,reactant_002,rxn_str,rxn_time,solvent_000,solvent_001,solvent_002,temperature,yield_000,yield_001
index,,,,,,,,,,,,,,,,,,,,,
123241,63385,CC(C)(C)OO,None,None,NaT,ord_dataset-feaf1b793c6d408aaec1cac7cc3ceadc,NaT,False,,CC(=O)CCCCn1c(=O)c2c(nc(CC(F)(F)F)n2C)n(C)c1=O,...,O=S(O)CC(F)(F)F,[Zn],None,18.0,O,Fc1c(F)c(F)c(C(F)(F)F)c(F)c1F,None,50.0,44.0,NaN
6546,27278,O=P([O-])([O-])[O-],[K+],None,NaT,ord_dataset-d92976309c3a48a3a64a4cf5e7048086,NaT,False,,Cc1cn(-c2ccc3n(c2=O)CCOC3=O)cn1,...,Cc1c[nH]cn1,None,None,18.0,CCC(C)(C)O,None,None,110.0,NaN,NaN
5477,535248,None,None,None,NaT,ord_dataset-5481550056a14935b76e031fb94b88be,NaT,False,,CC(C)n1nc(-c2ccc(F)c(O)c2)c2c(N)ncnc21,...,COc1cc(-c2nn(C(C)C)c3ncnc(N)c23)ccc1F,ClCCl,None,NaN,None,None,None,NaN,NaN,NaN
52638,53227,[Cu+2],O=S(=O)([O-])C(F)(F)F,None,NaT,ord_dataset-5c9a10329a8a48968d18879a48bb8ab2,NaT,False,Reactions were run in 8 x 30 mm glass vial ins...,COCCOc1ccccc1S(=O)(=O)N(c1ccc(C(F)(F)F)cc1)c1c...,...,OB(O)c1ccc(C(F)(F)F)cc1,None,None,18.0,CCOC(C)=O,CCN(CC)CC,None,60.0,0.0,2.11
16087,165126,None,None,None,NaT,ord_dataset-f65ca20d97ea49548e969f0e8a633933,NaT,False,,COCCN1C(=O)C(Nc2ccc(N3CCOCC3)cc2)=C(c2ccccc2)C1=O,...,CC#N,COCCN1C(=O)C(Cl)=C(c2ccccc2)C1=O,None,NaN,None,None,None,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35423,99235,None,None,None,NaT,ord_dataset-488402f6ec0d441ca2f7d6fabea7c220,NaT,False,,O=S1(=O)c2cc(C(F)(F)F)ccc2CC2CNCCN21,...,[H][H],O=C(OCc1ccccc1)N1CCN2C(Cc3ccc(C(F)(F)F)cc3S2(=...,None,NaN,None,None,None,NaN,NaN,NaN
36675,183794,None,None,None,NaT,ord_dataset-f65ca20d97ea49548e969f0e8a633933,NaT,False,,COc1cc2cc(Nc3cc(C)[nH]n3)nc(OC3CCC3)c2cc1C(=O)...,...,OC1CCC1,None,None,NaN,None,None,None,NaN,NaN,NaN
9346,240275,None,None,None,NaT,ord_dataset-0c1e1da868cd46c59efce6d5d4bc3d63,NaT,False,,Nc1ccc(C#Cc2cnc3ccc(-c4ccc(C(F)(F)F)cc4)cn23)cn1,...,Nc1ccc(I)cn1,None,None,NaN,None,None,None,NaN,NaN,NaN


In [4]:
retro_df

,original_index,agent_000,agent_001,agent_002,agent_003,agent_004,agent_005,agent_006,agent_007,agent_008,...,rxn_str,rxn_time,solvent_000,solvent_001,solvent_002,solvent_003,solvent_004,solvent_005,temperature,yield_000
index,,,,,,,,,,,,,,,,,,,,,
23318,97212,None,None,None,None,None,None,None,None,None,...,None,NaN,None,None,None,None,None,None,NaN,NaN
66417,421676,None,None,None,None,None,None,None,None,None,...,None,NaN,None,None,None,None,None,None,NaN,NaN
39620,529815,None,None,None,None,None,None,None,None,None,...,None,NaN,None,None,None,None,None,None,NaN,NaN
23591,267370,None,None,None,None,None,None,None,None,None,...,None,NaN,None,None,None,None,None,None,NaN,NaN
101184,35141,CC(=O)[O-],CCCC[N+](CCCC)(CCCC)CCCC,None,None,None,None,None,None,None,...,None,18.0,C1CCOC1,None,None,None,None,None,110.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67246,156648,None,None,None,None,None,None,None,None,None,...,None,NaN,None,None,None,None,None,None,NaN,NaN
59664,350921,None,None,None,None,None,None,None,None,None,...,None,NaN,None,None,None,None,None,None,NaN,NaN
70421,330435,None,None,None,None,None,None,None,None,None,...,None,NaN,None,None,None,None,None,None,NaN,NaN


In [5]:
forward_df.columns

Index(['original_index', 'agent_000', 'agent_001', 'agent_002',
       'date_of_experiment', 'extracted_from_file', 'grant_date', 'is_mapped',
       'procedure_details', 'product_000', 'product_001', 'reactant_000',
       'reactant_001', 'reactant_002', 'rxn_str', 'rxn_time', 'solvent_000',
       'solvent_001', 'solvent_002', 'temperature', 'yield_000', 'yield_001'],
      dtype='object')

In [6]:
retro_df.columns

Index(['original_index', 'agent_000', 'agent_001', 'agent_002', 'agent_003',
       'agent_004', 'agent_005', 'agent_006', 'agent_007', 'agent_008',
       'date_of_experiment', 'extracted_from_file', 'grant_date', 'is_mapped',
       'procedure_details', 'product_000', 'reactant_000', 'reactant_001',
       'rxn_str', 'rxn_time', 'solvent_000', 'solvent_001', 'solvent_002',
       'solvent_003', 'solvent_004', 'solvent_005', 'temperature',
       'yield_000'],
      dtype='object')

In [7]:
# concat if multiple reactants & products exist
# for forward
forward_df['reactant_list'] = forward_df.apply(lambda row: [row['reactant_000']] + ([row['reactant_001']] if pd.notna(row['reactant_001']) else []) + ([row['reactant_002']] if pd.notna(row['reactant_002']) else []), axis=1)
forward_df['product_list'] = forward_df.apply(lambda row: [row['product_000']] + ([row['product_001']] if pd.notna(row['product_001']) else []), axis=1)
# for retrosynthesis
retro_df['reactant_list'] = retro_df.apply(lambda row: [row['reactant_000']] + ([row['reactant_001']] if pd.notna(row['reactant_001']) else []), axis=1)
retro_df['product_list'] = retro_df.apply(lambda row: [row['product_000']], axis=1)

In [19]:
# Get reactants and products
forward_reactants = forward_df['reactant_list'].to_list()
forward_products = forward_df['product_list'].to_list()
retro_reactants = retro_df['reactant_list'].to_list()
retro_products = retro_df['product_list'].to_list()
print(len(forward_reactants),len(retro_reactants))

180230 111324


In [37]:
len(forward_products), len(retro_reactants)

(180230, 111324)

In [20]:
# Get InChI keys for deduplication
forward_inchi_set = load_mol_set('/data/data/Orderly/total_molset_forward.pkl')
retro_inchi_set = load_mol_set('/data/data/Orderly/total_molset_retro.pkl')
print(len(forward_inchi_set), len(retro_inchi_set))

996361 898867


In [21]:
def deduplicate_idx_by_inchi(input_reactants, output_products, check_inchi_set):
    """
    Deduplicate the input SMILES list by checking against the provided InChI set.
    """
    deduplicated_reactants = []
    deduplicated_products = []
    for i in tqdm(range(len(input_reactants))):
        frac_reactants = input_reactants[i]
        input_smiles = '.'.join(frac_reactants)
        mol = Chem.MolFromSmiles(input_smiles)
        if mol is not None:
            inchi_key = Chem.MolToInchiKey(mol)
            if inchi_key not in check_inchi_set:
                deduplicated_reactants.append(input_smiles)
                output_smiles = '.'.join(output_products[i])
                deduplicated_products.append(output_smiles)
    return deduplicated_reactants, deduplicated_products

In [22]:
ded_forward_reactants, ded_forward_products = deduplicate_idx_by_inchi(forward_reactants, forward_products, forward_inchi_set)

 61%|██████    | 109241/180230 [01:17<00:50, 1401.94it/s][12:27:34] WARNING: not removing hydrogen atom without neighbors
[12:27:34] WARNING: not removing hydrogen atom without neighbors
100%|██████████| 180230/180230 [02:07<00:00, 1409.70it/s]


In [23]:
len(ded_forward_reactants), len(ded_forward_products)

(79650, 79650)

In [32]:
ded_forward_reactants[2]

'COCCOc1ccccc1S(N)(=O)=O.OB(O)c1ccc(C(F)(F)F)cc1'

In [36]:
len(retro_reactants), len(retro_products)

(111324, 111324)

In [33]:
ded_retro_reactants, ded_retro_products = deduplicate_idx_by_inchi(retro_reactants, retro_products, retro_inchi_set)

100%|██████████| 111324/111324 [01:13<00:00, 1524.78it/s]


In [34]:
len(ded_retro_reactants), len(ded_retro_products)

(110570, 110570)

In [ ]:
for i in range(len(forward_reactants)):
    check_reactants = forward_reactants[i]
    combined_smiles = '.'.join(check_reactants)
    mol = Chem.MolFromSmiles(combined_smiles)

In [12]:
len(forward_reactants)

180230

In [13]:
len(retro_reactants)

111324

In [60]:
# Forward sample 1000, generate dataset
import random
import selfies as sf
random.seed(0)

forward_mol = []
forward_label = []
omitted_idx = []
total_len = len(ded_forward_reactants)
idx_lst = random.sample(range(total_len), 1000)

for i in idx_lst:
    smiles = ded_forward_reactants[i]
    label = ded_forward_products[i]
    try:
        mol = Chem.MolFromSmiles(smiles)
        label = sf.encoder(label)
        forward_label.append(label)
        forward_mol.append(mol)
    except:
        omitted_idx.append(i)
        continue
    
print(len(omitted_idx))

[13:06:46] WARNING: not removing hydrogen atom without neighbors


0


In [53]:
# Retrosynthesis sample 1000, generate dataset
import random
import selfies as sf
random.seed(42)

retro_mol = []
retro_label = []
omitted_idx = []
total_len = len(ded_retro_reactants)
idx_lst = random.sample(range(total_len), 1000)

for i in idx_lst:
    smiles = ded_retro_reactants[i]
    label = ded_retro_products[i]
    try:
        mol = Chem.MolFromSmiles(smiles)
        label = sf.encoder(label)
        retro_label.append(label)
        retro_mol.append(mol)
    except:
        omitted_idx.append(i)
        continue
    
print(len(omitted_idx))

0


In [51]:
def check_num_nodes(label_lst):
    num_nodes = []
    for label in label_lst:
        mol = sf.decoder(label)
        mol = Chem.MolFromSmiles(mol)
        graph = mol2graph(mol)
        num_nodes.append(graph['num_nodes'])
    print(np.mean(num_nodes), np.std(num_nodes))

In [61]:
check_num_nodes(forward_label)

24.905 10.68082276793319


In [54]:
check_num_nodes(retro_label)

25.234 9.448134419026859


In [62]:
list_orderly_forward_data = get_data_list(
    list_mol = forward_mol,
    list_label = forward_label,
    task = "orderly-forward_reaction_prediction",
    instruction_templates=instructions_smol.forward_reaction_prediction,
)

100%|██████████| 1000/1000 [00:01<00:00, 630.52it/s]


In [63]:
orderly_forward_te_dataset = datasets.Dataset.from_list(list_orderly_forward_data)
orderly_forward_te_dataset.save_to_disk(f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_orderly_forward_0509")

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [64]:
list_orderly_retro_data = get_data_list(
    list_mol = retro_mol,
    list_label = retro_label,
    task = "orderly-retrosynthesis",
    instruction_templates=instructions_smol.retrosynthesis,
)

100%|██████████| 1000/1000 [00:01<00:00, 656.46it/s]


In [65]:
orderly_retro_te_dataset = datasets.Dataset.from_list(list_orderly_retro_data)
orderly_retro_te_dataset.save_to_disk(f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_orderly_retro_0509")

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]